# T-340 — Critic, learned-router ABLATION (Qwen2.5-3B-Instruct)

**ResearchGPT**, ablation track. Asks whether a learned router beats the
Critic's arithmetic gates — and expects the answer to be **no**.

| | |
|---|---|
| Incumbent | `agents/critic.py` — **zero LLM calls**, every gate is arithmetic |
| Question | can a 3B model route `WRITE / RETRIEVE / REPLAN` better than the thresholds? |
| Serves | T-706 ablation 3 (− Critic loop) |

---

## 🔴 Stop: the training data does not exist

Measured across **all 78 runs in `outputs/runs/`**:

| decision | count |
|---|---|
| WRITE | 9 |
| RETRIEVE | 5 |
| REPLAN | 3 |
| **total** | **17** |

**Seventeen labelled examples for a three-class problem.** No fine-tune of any
size is meaningful on that, and a LoRA on 17 examples memorises them. Anything
this notebook trains would be reporting its own training set.

That is not a reason to delete the notebook — it is the **finding**, and it
should be written up as one. But it changes what the notebook can honestly do,
so what follows is structured as:

1. **§3** — measure the real data, and confirm it is too small (do not skip)
2. **§4** — build a synthetic set from the arithmetic gates themselves
3. **§5–7** — train and evaluate on it
4. **§8** — the verdict, whose expected value is written down in advance

### Why §4 proves less than it appears to

Generating training data *from the gate arithmetic* and then showing a model
learns it demonstrates only that a 3B network can fit three threshold
comparisons. Of course it can. It says **nothing** about whether the thresholds
are right, which is the only interesting question about the Critic.

So the honest use of this notebook is the reverse of the usual one: it exists to
document that **the deterministic design is not a shortcut but the correct
choice**, for two reasons the README already gives —

> a loop that spends tokens deciding whether to spend tokens is the worst place
> for a generative step

— and a second the numbers make sharper: at Groq's 200,000 tokens/day, every
Critic call is a call the Extractor does not get.

## 1 · Configuration

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
SEED         = 20260829

LORA_R       = 8             # deliberately small: the task is three thresholds
LORA_ALPHA   = 16
EPOCHS       = 2
LR           = 2e-4
BATCH        = 4
GRAD_ACCUM   = 4
MAX_SEQ_LEN  = 512

N_SYNTHETIC  = 3000
ADAPTER_DIR  = "/kaggle/working/critic-qwen3b"
TRACE_GLOB   = "/kaggle/input/researchgpt-traces/*/trace.jsonl"

# The live gates. Values from ReviewConfig in core/models.py; the ladder itself
# is Critic._decide in agents/critic.py. If either drifts, this notebook trains
# against a Critic the pipeline no longer has.
MIN_PAPERS                  = 20      # ReviewConfig.min_papers
COVERAGE_THRESHOLD          = 0.8     # ReviewConfig.coverage_threshold
GROUNDING_THRESHOLD         = 0.7     # ReviewConfig.grounding_threshold
MAX_ITERATIONS              = 3       # ReviewConfig.max_iterations
MIN_PAPERS_PER_SUB_QUESTION = 2       # agents/critic.py

## 2 · Dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install only what is absent. A re-run costs seconds, not minutes."""
    try:
        importlib.import_module(import_name or pkg.split("==")[0].split(">")[0].replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# transformers/torch/datasets ship with the Kaggle image; the PEFT stack does not.
for pkg, mod in [
    ("transformers>=4.44", "transformers"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("bitsandbytes", "bitsandbytes"),
    ("trl", "trl"),
]:
    ensure(pkg, mod)

import torch
print()
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("device     :", name, f"({vram:.1f} GB)")
    # T4 is sm_75: NO bf16. Every cell below uses fp16 for this reason.
    print("bf16 ok    :", torch.cuda.is_bf16_supported(), "  <- False on T4, as expected")

## 3 · The real data — measure it, then face it

Run this before anything else. If the count has grown well past 17 since
2026-08-29, this notebook's premise has changed and §4 should be replaced with
real supervision.

In [ ]:
import json, glob
from collections import Counter

real = []
for path in glob.glob(TRACE_GLOB):
    for line in open(path, encoding="utf-8", errors="replace"):
        try:
            d = json.loads(line)
        except Exception:
            continue
        if d.get("event") == "critic_decision":
            real.append(d.get("payload") or {})

print(f"critic_decision events across all runs: {len(real)}")
print("by decision:", dict(Counter(r.get("decision") for r in real)))
print()
if len(real) < 300:
    print("  VERDICT: too small to train on. Proceeding with synthetic data,")
    print("  which measures gate-fitting, NOT gate quality. See the header.")

## 4 · Synthetic gate-decision set

Sampled over the state space the Critic actually sees, and labelled by running
the real ladder. Deliberately includes **boundary cases** — states within ±0.02
of a threshold — because that is the only region where a learned router could
differ from arithmetic, and a uniform sample would hardly ever land there.

### ⚠️ The Critic is stateful, and that is fatal to a router

Transcribing `Critic._decide` exposes something the "just learn the routing"
framing hides. **Two of the three gates escalate based on what was decided
before:**

* Gate 1 (corpus below `min_papers`) returns RETRIEVE — *unless* a previous
  RETRIEVE already ran and left the corpus no bigger, in which case REPLAN.
* Gate 2 (coverage below threshold) returns RETRIEVE — *unless* a RETRIEVE has
  already been tried, in which case REPLAN.

So the same numeric state maps to **RETRIEVE or REPLAN depending on history**.
A stateless prompt of four numbers is not merely a worse router — it is
*unable* to express the policy, and the best it can do on ambiguous states is
guess the majority class. The prompt below therefore includes the prior
decisions, which is already an admission that the arithmetic version is the
simpler object.

In [ ]:
import random
random.seed(SEED)

WRITE, RETRIEVE, REPLAN = "WRITE", "RETRIEVE", "REPLAN"

def gate_label(iteration, n_papers, coverage, uncovered, grounding, prior, prior_papers):
    """Transcription of Critic._decide. A first-match ladder, in this order."""
    # Gate 0 - loop bound, checked first so nothing below can override it.
    if iteration >= MAX_ITERATIONS - 1:
        return WRITE
    # Gate 1 - corpus size, escalating to REPLAN if searching harder failed.
    if n_papers < MIN_PAPERS:
        if RETRIEVE in prior and n_papers <= prior_papers:
            return REPLAN
        return RETRIEVE
    # Gate 2 - coverage, escalating to REPLAN once a RETRIEVE has been tried.
    if coverage < COVERAGE_THRESHOLD or uncovered:
        if RETRIEVE in prior:
            return REPLAN
        return RETRIEVE
    # Gate 3 - grounding. None on the first pass: no draft exists yet.
    if grounding is not None and grounding < GROUNDING_THRESHOLD:
        return RETRIEVE
    return WRITE

def sample_state(boundary=False):
    if boundary:
        cov     = random.choice([COVERAGE_THRESHOLD, COVERAGE_THRESHOLD - 0.02,
                                 COVERAGE_THRESHOLD + 0.02])
        papers  = random.choice([MIN_PAPERS - 1, MIN_PAPERS, MIN_PAPERS + 1])
        grd     = random.choice([GROUNDING_THRESHOLD, GROUNDING_THRESHOLD - 0.02,
                                 GROUNDING_THRESHOLD + 0.02, None])
    else:
        cov     = round(random.random(), 3)
        papers  = random.randint(0, 600)
        grd     = random.choice([round(random.random(), 3), None])
    prior = random.choice([[], [RETRIEVE], [RETRIEVE, REPLAN], [REPLAN]])
    return {
        "iteration":    random.randint(0, MAX_ITERATIONS),
        "n_papers":     papers,
        "coverage":     round(cov, 3),
        # A sub-question under MIN_PAPERS_PER_SUB_QUESTION papers is "uncovered"
        # and trips gate 2 on its own, independently of the coverage ratio.
        "uncovered":    random.choice([[], [], ["SQ2"], ["SQ1", "SQ4"]]),
        "grounding":    grd,
        "prior":        prior,
        "prior_papers": random.randint(0, 600),
    }

PROMPT = (
    "Route the next step of a literature-review pipeline.\n"
    "Answer with one word: WRITE, RETRIEVE or REPLAN.\n\n"
    "iteration                 : {iteration} of " + str(MAX_ITERATIONS) + "\n"
    "papers retained           : {n_papers}\n"
    "sub-question coverage     : {coverage}\n"
    "under-covered sub-questions: {uncovered}\n"
    "grounding score           : {grounding}\n"
    "prior decisions           : {prior}\n"
    "papers at previous check  : {prior_papers}"
)

rows = []
for i in range(N_SYNTHETIC):
    st = sample_state(boundary=(i % 3 == 0))     # a third on the boundary
    rows.append({"state": st, "label": gate_label(**st)})
print("label mix:", dict(Counter(r["label"] for r in rows)))

split = int(len(rows) * 0.85)
train_rows, eval_rows = rows[:split], rows[split:]

In [ ]:
from datasets import Dataset

def to_chat(row):
    return {"text": tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(**row["state"])},
         {"role": "assistant", "content": row["label"]}],
        tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(to_chat)
eval_ds  = Dataset.from_list(eval_rows).map(to_chat)

## 5 · Load Qwen in 4-bit and attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    # fp16, not bf16 - the T4 is sm_75 and has no bf16 path. Using bf16 here
    # silently falls back to fp32 accumulation and doubles the step time.
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Causal LM + right padding corrupts the loss mask on the last token. Qwen's
# own chat template assumes left padding for generation.
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False           # incompatible with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # All seven projections. Attention-only (q,v) is the common shortcut and it
    # underfits format-learning tasks like this one, where the MLP carries most
    # of the "what shape is the output" signal.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=ADAPTER_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=20, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, fp16=True, optim="paged_adamw_8bit",
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LEN, dataset_text_field="text", packing=False,
    seed=SEED, report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=eval_ds, processing_class=tokenizer)
trainer.train()
trainer.save_model(ADAPTER_DIR)

## 7 · Evaluate — and price it

Two numbers decide this, and the second is the one that matters.

1. **Agreement with the arithmetic gates**, split out on boundary cases.
2. **Cost per decision.** The incumbent costs 0 tokens and ~0 ms. Anything above
   that has to be paid for out of a 200,000-token daily budget that the
   Extractor is already competing for.

In [ ]:
import torch, time
from sklearn.metrics import accuracy_score, classification_report

model.config.use_cache = True
model.eval()
LABELS = [WRITE, RETRIEVE, REPLAN]

gold, pred, lat = [], [], []
for row in eval_rows[:300]:
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(**row["state"])}],
        add_generation_prompt=True, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=6, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    lat.append(time.time() - t0)
    text = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip().upper()
    gold.append(row["label"])
    pred.append(next((l for l in LABELS if l in text), WRITE))

acc = accuracy_score(gold, pred)
print(f"  agreement with the gates   {acc:.1%}")
print(f"  median latency per decision {sorted(lat)[len(lat)//2]*1000:.0f} ms  (incumbent: ~0 ms)")
print(f"  tokens per decision         ~{MAX_SEQ_LEN//8} prompt + 6 out  (incumbent: 0)")
print()
print(classification_report(gold, pred, labels=LABELS, zero_division=0))

## 8 · Verdict — written before the numbers

**Reject the learned router unless it disagrees with the gates in a way that is
demonstrably better on real runs.** High agreement is *not* an argument for
adoption: a model that agrees 99% of the time has reproduced arithmetic at a
cost of tokens and latency, which is strictly worse than the arithmetic.

The only result that would justify a change is a *disagreement* that, checked
against real run outcomes, routes better than the thresholds did. With 17 real
decisions on record, that cannot be established here.

**What to write in the report (T-706):**

> A learned routing policy was trained as an ablation and rejected. It
> reproduced the deterministic gates at N% agreement while adding a per-decision
> LLM call to a pipeline already bounded by a 200,000-token daily budget. The
> deterministic Critic is retained: its gates are auditable, cost nothing, and
> cannot hallucinate a routing decision. The ablation's real finding is that the
> project has only 17 recorded routing decisions, which is itself a limitation
> of the evaluation rather than of the design.

In [ ]:
print("VERDICT: REJECT - the deterministic Critic stands.")
print()
print(f"  agreement       {acc:.1%}   (reproducing arithmetic is not an argument for adoption)")
print(f"  added cost      ~{sorted(lat)[len(lat)//2]*1000:.0f} ms and ~{MAX_SEQ_LEN//8} tokens per decision")
print(f"  real decisions  {len(real)}   (need ~300+ to argue this empirically)")